# فصل چهارم: آزمایش‌های پالایش حوزه فرکانس

این Notebook مکمل فصل چهارم است و از داده مصنوعی مستقل استفاده می‌کند. برای اجرای مجدد به NumPy، Matplotlib و Pillow نیاز دارید.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

N=256
y,x=np.indices((N,N))
image=0.15+0.3*x/N
image[((x-85)**2+(y-110)**2)<42**2]=0.85
image[(x>155)&(x<215)&(y>153)&(y<210)]=0.25
plt.imshow(image,cmap="gray"); plt.title("Synthetic input"); plt.axis("off");

## ۱. طیف، اندازه و فاز

فاز حامل اطلاعات مربوط به محل قرارگیری ساختارهاست. اندازه طیف را برای نمایش به‌صورت لگاریتمی رسم می‌کنیم.

In [ ]:
F=np.fft.fftshift(np.fft.fft2(image))
fig,ax=plt.subplots(1,3,figsize=(12,4))
for a,t,data in zip(ax,["Image","Log magnitude","Phase"],[image,np.log1p(np.abs(F)),np.angle(F)]):
 a.imshow(data,cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

## ۲. سه پایین‌گذر و یک بالاگذر

پارامتر cutoff و مرتبه باتروورث را تغییر دهید و ringing و محوشدگی را مقایسه کنید.

In [ ]:
yy=np.fft.fftshift(np.fft.fftfreq(N))*N
D=np.hypot(*np.meshgrid(yy,yy))
D0=22
filters={"Ideal":(D<=D0).astype(float),"Gaussian":np.exp(-D**2/(2*D0**2)),"Butterworth":1/(1+(D/D0)**6)}
fig,ax=plt.subplots(1,3,figsize=(12,4))
for a,(label,H) in zip(ax,filters.items()):
 z=np.fft.ifft2(np.fft.ifftshift(F*H)).real
 a.imshow(z,cmap="gray");a.set_title(label);a.axis("off")
plt.show()

## ۳. نویز تناوبی و ناتچ‌های متقارن

در این آزمایش یک موج سینوسی به تصویر اضافه می‌کنیم و دو نقطه متقارن طیف را تضعیف می‌کنیم.

In [ ]:
noisy=image+0.2*np.sin(2*np.pi*20*x/N)
S=np.fft.fftshift(np.fft.fft2(noisy));H=np.ones_like(image)
for dx in [-20,20]:
 d=np.hypot(x-(N//2+dx),y-N//2)
 H*=1-np.exp(-d**2/(2*3**2))
restored=np.fft.ifft2(np.fft.ifftshift(S*H)).real
fig,ax=plt.subplots(1,3,figsize=(11,4))
for a,t,z in zip(ax,["Interference","Notch","Restored"],[noisy,H,restored]):a.imshow(z,cmap="gray");a.set_title(t);a.axis("off")
plt.show()

## پرسش

چرا در نویز سینوسی به دو ناتچ متقارن نیاز است؟ اگر عرض ناتچ را دو برابر کنیم چه مقدار جزئیات واقعی نیز حذف می‌شود؟